In [ ]:
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.mask import mask
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from typing import Tuple, List, Dict
from tqdm import tqdm
import os
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.ndimage import zoom
import numpy as np
import numpy as np
import random
import json
from pathlib import Path
from typing import Dict, List, Tuple
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from typing import Dict, Tuple
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
from pathlib import Path

from data_loader import (
    load_dem, load_infiltration_map, load_landuse_map, print_raster_stats,gmm_no_manila_path_box_load_all_flood_maps,
    load_all_rainfall_scenarios, get_rainfall_stats, visualize_raster, visualize_flood_maps_grid, manila_path_shape_load_all_flood_maps,
    get_raster_stats, load_manila_mask, gmm_path_load_all_flood_maps
)

from data_preprocessing import (normalize_dem, normalize_infiltration, normalize_landuse, handle_nodata )

from flood_maps import (FLOOD_CLASSES, reconstruct_map_from_patches, categorize_all_flood_maps_patch, visualize_all_categorized_maps)

from quadrant import ( stack_spatial_inputs, split_into_quadrants, build_train_test_split, extract_spatial_patches, get_quadrant_patch_indices, visualize_quadrant_split )

from vit import ViT

from training import FloodPatchDataset, train_kfold

RANDOM_SEED = 42
NUM_SCENARIO = 50
PATCH_SIZE = 4

In [ ]:
repo_root = os.getcwd()
gmm_path = os.path.join(repo_root, "COP-30m-GMM")
gmm_no_manila_path = os.path.join(repo_root, "COP-30m-GMM-ManilaMaskOut")
manila_path = os.path.join(repo_root, "COP-30m-ManilaOnly")
rainfall_path = os.path.join(repo_root, "mm_hr_scenarios")

# Loading data

In [ ]:
# Load DEM
print("\n--- Loading DEM ---")
dem_path = os.path.join(gmm_path, "greater_mm_bbox_dem_cop.tif")
if os.path.exists(dem_path):
    dem_data, dem_meta = load_dem(dem_path)
    dem_stats = get_raster_stats(dem_data, "DEM")
    print_raster_stats(dem_stats)
else:
    print(f"WARNING: DEM file not found at {dem_path}")
    dem_data = None

fig1 = visualize_raster(dem_data, title="DEM - Manila", cmap='terrain')

# Load Infiltration Map
print("\n--- Loading Infiltration Map ---")
infilt_path = os.path.join(gmm_path, "GM_Infilt_fixed.tif")
if os.path.exists(infilt_path):
    infilt_data, infilt_meta = load_infiltration_map(infilt_path)
    infilt_stats = get_raster_stats(infilt_data, "Infiltration Map")
    print_raster_stats(infilt_stats)
else:
    print(f"WARNING: Infiltration file not found at {infilt_path}")
    infilt_data = None
fig2 = visualize_raster(infilt_data, title="Infiltration Map", cmap='YlGnBu')

# Load Landuse Map
print("\n--- Loading Landuse Map ---")
landuse_path =  os.path.join(gmm_path, "GM_LU.tif")
if os.path.exists(landuse_path):
    landuse_data, landuse_meta = load_landuse_map(landuse_path)
    landuse_stats = get_raster_stats(landuse_data, "Landuse Map")
    print_raster_stats(landuse_stats)
else:
    print(f"WARNING: Landuse file not found at {landuse_path}")
    landuse_data = None
    
fig3 = visualize_raster(landuse_data, title="Landuse Map", cmap='tab20')

# Load all flood maps
print("\n--- Loading Flood Maps (Ground Truth) ---")
fm_path = os.path.join(gmm_path, "exp-des-4")
flood_maps, flood_metadata = gmm_path_load_all_flood_maps(gmm_path, num_scenarios=NUM_SCENARIO)

if len(flood_maps) > 0:
    print(f"\nFlood maps successfully loaded: {len(flood_maps)}")
    
    if len(flood_maps) >= NUM_SCENARIO:
        flood_stats = get_raster_stats(flood_maps[-1], "Flood Map RS20")
else:
    print("WARNING: No flood maps loaded!")

fig4 = visualize_flood_maps_grid(flood_maps=flood_maps,scenario_ids=list(range(1, 51)),figsize=(25, 20),ncols=5,cmap='terrain')
print_raster_stats(flood_stats)

In [ ]:
print("LOADING RAINFALL SCENARIOS")
# Load rainfall scenarios
rainfall_scenarios = load_all_rainfall_scenarios(rainfall_path)

if len(rainfall_scenarios) > 0:
    print(f"\nRainfall scenarios successfully loaded: {len(rainfall_scenarios)}")
    
    if len(rainfall_scenarios) >= NUM_SCENARIO:
        rain_stats = get_rainfall_stats(rainfall_scenarios[-1], scenario_id=20)
else:
    print("WARNING: No rainfall scenarios loaded!")

display(rainfall_scenarios)

# Data Preprocessing

In [ ]:
print("\nProcessing DEM...")
dem_clean = handle_nodata(dem_data, nodata_value=dem_meta.get('nodata'), fill_method='mean')

print("\nProcessing Infiltration...")
infilt_clean = handle_nodata(infilt_data, nodata_value=infilt_meta.get('nodata'), fill_method='mean')

print("\nProcessing Landuse...")
landuse_clean = handle_nodata(landuse_data, nodata_value=landuse_meta.get('nodata'), fill_method='interpolate')

In [ ]:
# Normalize DEM
print("\nNormalizing DEM...")
dem_normalized, dem_params = normalize_dem(dem_data, method='minmax')
# Verify normalization
dem_norm_stats = get_raster_stats(dem_normalized, "DEM (Normalized)")
print_raster_stats(dem_norm_stats)

# Normalize Infiltration
print("\nNormalizing Infiltration...")
infilt_normalized, infilt_params = normalize_infiltration(infilt_data)
# Verify normalization
infilt_norm_stats = get_raster_stats(infilt_normalized, "Infiltration (Normalized)")
print_raster_stats(infilt_norm_stats)

# Normalize Landuse
print("\nNormalizing Landuse...")
landuse_normalized, landuse_params = normalize_landuse(landuse_data, method='minmax')
# Verify normalization
landuse_norm_stats = get_raster_stats(landuse_normalized, "Landuse (Normalized)")
print_raster_stats(landuse_norm_stats)


# Flood Maps (Ground Truths)

In [ ]:
flood_maps_categorized_patch, patch_metadata = categorize_all_flood_maps_patch(flood_maps, patch_size=PATCH_SIZE, stride=None, categorization_method='majority', verbose=True)

In [ ]:
reconstructed_maps = []
for patch_labels, metadata in zip(flood_maps_categorized_patch, patch_metadata):
    reconstructed = reconstruct_map_from_patches(patch_labels, metadata, method='nearest')
    reconstructed_maps.append(reconstructed)
fig_all = visualize_all_categorized_maps(reconstructed_maps, scenario_ids=list(range(1, 51)), figsize=(5, 10), ncols=5)
plt.show()

In [ ]:
display(flood_maps_categorized_patch)

# Data Splitting Strategy

In [ ]:
print("Stack Spatial Inputs")
spatial_stack = stack_spatial_inputs(dem_normalized, infilt_normalized, landuse_normalized)

print("Flood maps already categorized →", len(flood_maps_categorized_patch), "scenarios")

print("Quadrant Split — Spatial Stack")
spatial_quadrants = split_into_quadrants(spatial_stack)

print("\nExtract Spatial Patches (full map)")
spatial_patches = extract_spatial_patches(spatial_stack, patch_size=PATCH_SIZE)

print("\nCompute Quadrant Patch Indices")
H, W = dem_normalized.shape
quadrant_indices = get_quadrant_patch_indices(H, W, patch_size=PATCH_SIZE)

print("\nBuild Train / Test Split")
dataset = build_train_test_split(spatial_patches, flood_maps_categorized_patch, quadrant_indices, rainfall_scenarios=rainfall_scenarios,)

In [ ]:
visualize_quadrant_split(dem_normalized)

# Rainfall Encoding

In [ ]:
all_intensities = np.concatenate([df['intensity_mmhr'].values for df in rainfall_scenarios])
rain_min, rain_max = all_intensities.min(), all_intensities.max()

def normalize_rainfall_scenarios(rainfall_scenarios, rain_min, rain_max):
    normalized = []
    for df in rainfall_scenarios:
        intensity = (df['intensity_mmhr'].values - rain_min) / (rain_max - rain_min)
        normalized.append(intensity.astype(np.float32))
    return np.stack(normalized, axis=0) 

rain_embeddings = normalize_rainfall_scenarios(rainfall_scenarios, rain_min, rain_max)
print(rain_embeddings.shape) 

# ViT Config

In [ ]:
from config import Config
from model_config import model_factory, make_model
from training import FloodPatchDataset, train_kfold

cfg = Config()  
cfg.save(cfg.train.output_dir / 'config.json')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_dataset = FloodPatchDataset(
    X_spatial = dataset['X_train'],
    y_labels  = dataset['y_train'],
    rainfall  = rain_embeddings,
)

# Training Setup

In [ ]:
# ── K-Fold Training ─────────────────────────────────────────────────────
fold_histories, best_fold = train_kfold(
    model_factory = model_factory(cfg.model),
    full_dataset  = train_dataset,
    y_train       = dataset['y_train'],
    cfg           = cfg.train,
    device        = device,
)

In [ ]:
# Loading best model
ckpt  = torch.load(cfg.train.output_dir / f'fold_{best_fold+1}' / 'best_model.pt', map_location=device)
model = make_model(cfg.model)
model.load_state_dict(ckpt['model_state_dict'])
model.to(device).eval()
print(f"✓ Loaded fold {best_fold+1}  (val_acc={ckpt['best_val_acc']:.4f})")

# Testing

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
from testing import ( prepare_ground_truth, evaluate_all_scenarios, plot_metrics_summary, plot_per_scenario_metrics,plot_per_class_metrics, plot_flood_on_dem )

print("\nPreparing ground truth maps for test scenarios...")
ground_truth = prepare_ground_truth(
    y_test           = dataset['y_test'],     
    quadrant_indices = quadrant_indices,
    patch_metadata   = patch_metadata[0],
)

In [ ]:
print("\nEvaluating all test scenarios...")
results = evaluate_all_scenarios(
    model           = model,
    X_test          = dataset['X_test'],
    y_test          = dataset['y_test'],
    rain_normalized = rain_embeddings,
    ground_truth    = ground_truth,
    batch_size      = 512,
    device          = device,
    num_classes     = cfg.model.num_classes,
)

In [ ]:
print("\nGenerating visualizations...")

fig1 = plot_metrics_summary(results)
fig1.savefig("eval_metrics_summary.png", dpi=150, bbox_inches='tight')

fig2 = plot_per_scenario_metrics(results)
fig2.savefig("eval_per_scenario.png", dpi=150, bbox_inches='tight')

fig3 = plot_per_class_metrics(results)
fig3.savefig("eval_per_class.png", dpi=150, bbox_inches='tight')

plt.show()

In [ ]:
H_dem, W_dem = dem_clean.shape
dem_q3 = dem_clean[H_dem//2:, :W_dem//2]   

best_idx  = results['aggregate']['best_scenario']  - 1
worst_idx = results['aggregate']['worst_scenario'] - 1

# Single scenario — predicted
fig4 = plot_flood_on_dem(
    dem_q3      = dem_q3,
    flood_map   = results['pred_maps'][best_idx],
    scenario_id = results['aggregate']['best_scenario'],
    metrics     = results['per_scenario'][best_idx],
    mode        = 'pred',
    inset_center= (150, 420),
    figsize     = (9, 11),
    inset_size  = 60
)
fig4.savefig("eval_dem_best_pred.png", dpi=150, bbox_inches='tight')

# Single scenario — ground truth
fig5 = plot_flood_on_dem(
    dem_q3      = dem_q3,
    flood_map   = results['gt_maps'][best_idx],
    scenario_id = results['aggregate']['best_scenario'],
    metrics     = results['per_scenario'][best_idx],
    mode        = 'gt',
    inset_center=(150, 420),
    figsize     = (9, 11),
    inset_size  = 60
)
fig5.savefig("eval_dem_best_gt.png", dpi=150, bbox_inches='tight')

plt.show()
print("\nAll figures saved.")
print(f"  Best  scenario: RS{results['aggregate']['best_scenario']}")
print(f"  Worst scenario: RS{results['aggregate']['worst_scenario']}")

In [ ]:
fig6 = plot_flood_on_dem(
    dem_q3      = dem_q3,
    flood_map   = results['pred_maps'][worst_idx],
    scenario_id = results['aggregate']['worst_scenario'],
    metrics     = results['per_scenario'][worst_idx],
    mode        = 'pred',
    inset_center= (150, 420),
    figsize     = (9, 11),
    inset_size  = 60
)
fig6.savefig("eval_dem_worst_pred.png", dpi=150, bbox_inches='tight')

# Single scenario — ground truth
fig7 = plot_flood_on_dem(
    dem_q3      = dem_q3,
    flood_map   = results['gt_maps'][worst_idx],
    scenario_id = results['aggregate']['worst_scenario'],
    metrics     = results['per_scenario'][worst_idx],
    mode        = 'gt',
    inset_center=(150, 420),
    figsize     = (9, 11),
    inset_size  = 60
)
fig7.savefig("eval_dem_worst_gt.png", dpi=150, bbox_inches='tight')